# SIH26184: Predictive Cash-Withdrawal Location Intelligence
## Notebook 2: Main Model Development & Chronological Tuning (v2.0.0)
---
This notebook covers:
1. Fitting LightGBM Gradient Boosted Decision Trees on the chronological training set (<= 2022)
2. Early stopping optimization on out-of-time validation set (2023 H1)
3. Integration of 30 dynamic features (incorporating RBI Monthly Banking & NCRB Crime Context)
4. Out-of-time test set evaluation comparing against Progressive Baselines
5. Packaging model artifacts to artifacts/model/ and models/v2.0.0/


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
from src.features.feature_pipeline import FeaturePipeline
from src.models.main_model import MainModelTrainer
from src.evaluation.metrics import format_metrics_table


In [2]:
df = pd.read_parquet('../data/synthetic/candidate_dataset.parquet')
train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

print(f'Training Set: {len(train_df)} rows | Validation Set: {len(val_df)} rows | Test Set: {len(test_df)} rows')

trainer = MainModelTrainer(version='v2.0.0')
trainer.pipeline = FeaturePipeline.load(artifact_dir='../artifacts/model', version='v2.0.0')
model = trainer.train(train_df, val_df)
test_metrics, scores = trainer.evaluate(test_df)

print('\n--- Main Model v2.0.0 Test Metrics ---')
print(format_metrics_table({'Main Model (LightGBM v2.0.0)': test_metrics}))


Training Set: 16460 rows | Validation Set: 4174 rows | Test Set: 6140 rows
Fitting FeaturePipeline (v2.0.0) on training data...
Fitting LightGBM model with parameters: {'objective': 'binary', 'metric': 'average_precision', 'boosting_type': 'gbdt', 'n_estimators': 400, 'learning_rate': 0.035, 'num_leaves': 31, 'max_depth': 6, 'subsample': 0.85, 'colsample_bytree': 0.85, 'scale_pos_weight': 12.0, 'random_state': 42, 'verbose': -1}

--- Main Model v2.0.0 Test Metrics ---
| Model | Hit@1 | Hit@3 | Hit@5 | Hit@10 | MRR | NDCG@5 | PR-AUC | ROC-AUC |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Main Model (LightGBM v2.0.0) | 6.2% | 14.8% | 26.7% | 55.7% | 0.1897 | 0.1596 | 0.0393 | 0.5180 |
